In [ ]:
# %pip install streamlit

  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
   ---------------------------------------- 0.0/9.0 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.0 MB 6.7 MB/s eta 0:00:02
   ---------- ----------------------------- 2.4/9.0 MB 5.8 MB/s eta 0:00:02
   --------------- ------------------------ 3.4/9.0 MB 5.9 MB/s eta 0:00:01
   -------------------- ------------------- 4.7/9.0 MB 5.7 MB/s eta 0:00:01
   ------------------------- -------------- 5.8/9.0 MB 5.7 MB/s eta 0:00:01
   --------------------------- ------------ 6.3/9.0 MB 5.2 MB/s eta 0:00:01
   -------------------------------- ------- 7.3/9.0 MB 5.1 MB/s eta 0:00:01
   ------------------------------------- -- 8.4/9.0 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 9.0/9.0 MB 5.2 MB/s  0:00:01
   ---------------------------------------- 0.0/795.4 kB ? eta -:--:--
   ---------------------------------------- 795.4/795.4 kB 5.7 MB/s  0:00:00
   ----------------------------------

UsageError: Line magic function `%streamlit` not found.


In [ ]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()


def get_ai_message(user_message):

    embedding = UpstageEmbeddings(model='embedding-passage')
    index_name = 'tax-index-upstage'
    database = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embedding)

    llm = ChatAnthropic(
        model='claude-3-haiku-20240307',
        temperature=0.3,
        top_p=1
    )

    retriever = database.as_retriever(
        search_kwargs={'k': 20, 'fetch_k': 50}
    )
    
        
    # 사전 정의
    dictionary = ['사람을 나타내는 표현 -> 거주자']
    kward_prompt = ChatPromptTemplate.from_template(f'''                                              
        사용자의 질문 중 우리의 사전을 참고하여 사용자의 질문 속 특정 어휘만을 수정해주세요.
        만일 변경할 필요가 없다고 판단된다면 입력된 질문을 그대로 반환해주세요.
        ---
        사전: {dictionary}
        질문: {{question}} 
    ''')      
    
    # 용어 필터 chain 만들기
    dict_chain = kward_prompt | llm | StrOutputParser()

    rag_prompt = ChatPromptTemplate.from_messages([
        ("system", """다음 context를 바탕으로 질문에 답하세요.
        당신은 대한민국 세법 전문가입니다. 다음 Context를 바탕으로 사용자의 질문에 대해 명확하고 정확하게 답변해주세요.
        
        [답변 작성 규칙]
        1. 질문에 대한 핵심 내용만 간결하게 요약해서 답변하세요.
        2. n단계 모형을 설명할 때는 1단계부터 n단계까지 번호를 매겨서 명확히 구분하세요.
        3. 각 단계 설명은 1~2문장으로 짧게 요약하세요.
        4. 문서를 꼼꼼히 확인하고 사실과 다른 내용은 지어내지 마세요.
        
        다음은 Few-shot 예시입니다. 
        question: 확신유형의 보증과 용역유형의 보증에 대해 설명해줘.
        answer: 확신유형의 보증은 수행의무로 회계처리하지 않고 용역유형의 보증은 수행의무로 회계처리한다.
        
        Let's think step by step
        
        \n\nContext: {context}"""),
        ("human", "{question}")
    ])

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    tax_qa_chain = (
        dict_chain |
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | rag_prompt
        | llm
        | StrOutputParser()
    )
    
    ai_message = tax_qa_chain.invoke({'question': user_message})
    return ai_message


In [ ]:
import llm

def get_ai_message(user_message, llm=None, rag_prompt=None, dictionary_chain=None):    
    retriever = get_retriever()
    if llm is None:
        llm = get_claude()    
    if dictionary_chain is None:
        dictionary_chain = get_dictionary_chain()
    if rag_prompt is None:
        rag_prompt = get_tax_prompt()

    
    tax_qa_chain = (
        dictionary_chain |
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | rag_prompt
        | llm
        | StrOutputParser()
    )
    
    ai_message = tax_qa_chain.invoke({'question': user_message})
    return ai_message

c:\Users\16Z90P\anaconda3\envs\langchain-basics\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from langchain_upstage import UpstageEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder



# 세션별 히스토리 저장소
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


def get_claude(model='claude-3-haiku-20240307', temperature=0.05, top_p=1, max_tokens=None):
    claude = ChatAnthropic(
        model=model,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )
    return claude


def get_retriever(index_name=None, embedding=None, k=20, fetch_k=100):
    if index_name is None:
        index_name = 'tax-index-upstage'
    if embedding is None:
        embedding = UpstageEmbeddings(model='embedding-passage')

    database = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embedding)
    retriever = database.as_retriever(search_kwargs={'k': k, 'fetch_k': fetch_k})
    return retriever


def get_dictionary_chain(llm=None, dictionary=None):
    if llm is None:
        llm = get_claude()
    if dictionary is None:
        dictionary = ['사람을 나타내는 표현 -> 거주자']
    
    keyword_prompt = ChatPromptTemplate.from_template(
        f'''사용자의 질문 중 우리의 사전을 참고하여 사용자의 질문 속 특정 어휘만을 수정해주세요.
        만일 변경할 필요가 없다고 판단된다면 입력된 질문을 그대로 반환해주세요.
        ---
        사전: {dictionary}
        질문: {{question}}'''
    )
    
    dict_chain = keyword_prompt | llm | StrOutputParser()
    return dict_chain


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# 히스토리 기반 질문 재작성 체인
def get_contextualize_chain(llm=None):
    if llm is None:
        llm = get_claude()
    
    contextualize_prompt = ChatPromptTemplate.from_messages([
        ("system", 
         """대화 기록과 최신 사용자 질문을 보고, 대화 기록 없이도 이해할 수 있는 독립적인 질문으로 재작성하세요.
         질문에 답하지 말고, 필요하면 재작성만 하고 필요 없으면 그대로 반환하세요."""),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    
    contextualize_chain = contextualize_prompt | llm | StrOutputParser()
    return contextualize_chain

# Few-shot 추가 필요
def get_tax_prompt():
    rag_prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 대한민국 세법 전문가입니다. 
        다음 Context를 바탕으로 사용자의 질문에 답변해주세요.
        
        [답변 작성 규칙]
        1. 질문에 대한 핵심 내용만 간결하게 요약해서 답변하세요.
        2. n단계 모형을 설명할 때는 1단계부터 n단계까지 번호를 매겨서 명확히 구분하세요.
        3. 각 단계 설명은 1~2문장으로 짧게 요약하세요.
        4. 문서를 꼼꼼히 확인하고 사실과 다른 내용은 지어내지 마세요.
        
        Let's think step by step
        ---
        Context: {context}"""),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    return rag_prompt


def get_rag_chain(llm=None, retriever=None, contextualize_chain=None, rag_prompt=None):
    if llm is None:
        llm = get_claude()
    if retriever is None:
        retriever = get_retriever()
    if contextualize_chain is None:
        contextualize_chain = get_contextualize_chain(llm)
    if rag_prompt is None:
        rag_prompt = get_tax_prompt()
    
    def contextualize_and_retrieve(input_dict):
        # 히스토리가 있으면 질문 재작성, 없으면 그대로
        chat_history = input_dict.get("chat_history", [])
        user_input = input_dict["input"]
        
        if chat_history:
            contextualized_q = contextualize_chain.invoke({
                "input": user_input,
                "chat_history": chat_history
            })
        else:
            contextualized_q = user_input
        
        # 재작성된 질문으로 검색
        docs = retriever.invoke(contextualized_q)
        return format_docs(docs)
    
    # RAG 체인 조립
    rag_chain = (
        RunnablePassthrough.assign(
            context=RunnableLambda(contextualize_and_retrieve)
        )
        | rag_prompt
        | llm
        | StrOutputParser()
    )
    
    # 히스토리 래핑
    conversational_rag_chain = RunnableWithMessageHistory(
        rag_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="chat_history",
    )
    
    return conversational_rag_chain


def get_ai_message(user_message, session_id="abc123"):
    # 1. 용어 변환
    dictionary_chain = get_dictionary_chain()
    refined_question = dictionary_chain.invoke({"question": user_message})
    
    # 2. RAG + 히스토리
    rag_chain = get_rag_chain()
    ai_message = rag_chain.invoke(
        {"input": refined_question},
        config={"configurable": {"session_id": session_id}}
    )
    
    return ai_message